### Packages

In [1]:
!pip install arch

     |████████████████████████████████| 798kB 2.7MB/s 


In [2]:
!pip install pmdarima

     |████████████████████████████████| 1.5MB 2.7MB/s 
     |████████████████████████████████| 2.1MB 10.3MB/s 
     |████████████████████████████████| 8.7MB 24.8MB/s 
  Found existing installation: Cython 0.29.21
    Uninstalling Cython-0.29.21:
      Successfully uninstalled Cython-0.29.21
  Found existing installation: statsmodels 0.10.2
    Uninstalling statsmodels-0.10.2:
      Successfully uninstalled statsmodels-0.10.2


In [3]:
!pip install yfinance

  Created wheel for yfinance: filename=yfinance-0.1.54-py2.py3-none-any.whl size=22409 sha256=3beddd1a54cb367d083e950baee1464fdb87b28badad257aba0c4b310790de98
  Stored in directory: /root/.cache/pip/wheels/f9/e3/5b/ec24dd2984b12d61e0abf26289746c2436a0e7844f26f2515c
Successfully built yfinance


In [3]:
!pip list

Package                 Version
----------------------- -----------
arch                    7.2.0
asttokens               3.0.0
beautifulsoup4          4.13.4
certifi                 2025.7.14
cffi                    1.17.1
charset-normalizer      3.4.2
colorama                0.4.6
comm                    0.2.2
contourpy               1.3.2
curl_cffi               0.12.0
cycler                  0.12.1
Cython                  3.1.2
debugpy                 1.8.14
decorator               5.2.1
executing               2.2.0
fonttools               4.58.4
frozendict              2.4.6
idna                    3.10
ipykernel               6.29.5
ipython                 9.3.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
joblib                  1.5.1
jupyter_client          8.6.3
jupyter_core            5.8.1
kiwisolver              1.4.8
matplotlib              3.10.3
matplotlib-inline       0.1.7
multitasking            0.0.12
nest-asyncio            1.6.0
numpy              

In [1]:
import numpy as np
import pandas as pd
import scipy
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model
import yfinance
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set()
import pmdarima as pm
from pmdarima.model_selection import train_test_split

### Loading the data

In [2]:
raw_data = yfinance.download (tickers = "^GSPC ^FTSE ^N225 ^GDAXI", start = "1994-01-07", end = "2020-03-20", 
                              interval = "1d", group_by = 'ticker', auto_adjust = True)

[*********************100%***********************]  4 of 4 completed


In [3]:
df_comp = raw_data.copy()

In [4]:
df_comp.head()

Ticker             ^N225                                                   \
Price               Open          High           Low         Close Volume   
Date                                                                        
1994-01-07  17842.980469  18131.410156  17787.480469  18124.009766    0.0   
1994-01-10  18186.519531  18567.060547  18186.519531  18443.439453    0.0   
1994-01-11  18481.849609  18671.669922  18373.039062  18485.250000    0.0   
1994-01-12  18447.339844  18807.080078  18301.929688  18793.880859    0.0   
1994-01-13  18770.380859  18823.380859  18548.750000  18577.259766    0.0   

Ticker            ^FTSE                                                \
Price              Open         High          Low        Close Volume   
Date                                                                    
1994-01-07  3401.399902  3446.800049  3398.699951  3446.000000    0.0   
1994-01-10  3465.699951  3468.100098  3430.000000  3440.600098    0.0   
1994-01-11  3442.500000  3442.500000  3413.500000  3413.800049    0.0   
1994-01-12  3394.800049  3402.399902  3372.000000  3372.000000    0.0   
1994-01-13  3380.699951  3383.300049  3356.899902  3360.000000    0.0   

Ticker           ^GSPC                                                   \
Price             Open        High         Low       Close       Volume   
Date                                                                      
1994-01-07  467.089996  470.260010  467.029999  469.899994  324920000.0   
1994-01-10  469.899994  475.269989  469.549988  475.269989  319490000.0   
1994-01-11  475.269989  475.279999  473.269989  474.130005  305490000.0   
1994-01-12  474.130005  475.059998  472.140015  474.170013  310690000.0   
1994-01-13  474.170013  474.170013  471.799988  472.470001  277970000.0   

Ticker           ^GDAXI                                                
Price              Open         High          Low        Close Volume  
Date                                                                   
1994-01-07  2218.959961  2227.639893  2201.820068  2224.949951    0.0  
1994-01-10  2231.840088  2238.010010  2222.000000  2225.000000    0.0  
1994-01-11  2225.429932  2235.610107  2225.179932  2228.100098    0.0  
1994-01-12  2227.120117  2227.790039  2182.060059  2182.060059    0.0  
1994-01-13  2171.500000  2183.709961  2134.100098  2142.370117    0.0

In [5]:
df_comp['spx'] = df_comp['^GSPC'].Close[:]
df_comp['dax'] = df_comp['^GDAXI'].Close[:]
df_comp['ftse'] = df_comp['^FTSE'].Close[:]
df_comp['nikkei'] = df_comp['^N225'].Close[:] 

In [6]:
df_comp.head()

Ticker             ^N225                                                   \
Price               Open          High           Low         Close Volume   
Date                                                                        
1994-01-07  17842.980469  18131.410156  17787.480469  18124.009766    0.0   
1994-01-10  18186.519531  18567.060547  18186.519531  18443.439453    0.0   
1994-01-11  18481.849609  18671.669922  18373.039062  18485.250000    0.0   
1994-01-12  18447.339844  18807.080078  18301.929688  18793.880859    0.0   
1994-01-13  18770.380859  18823.380859  18548.750000  18577.259766    0.0   

Ticker            ^FTSE                                                ...  \
Price              Open         High          Low        Close Volume  ...   
Date                                                                   ...   
1994-01-07  3401.399902  3446.800049  3398.699951  3446.000000    0.0  ...   
1994-01-10  3465.699951  3468.100098  3430.000000  3440.600098    0.0  ...   
1994-01-11  3442.500000  3442.500000  3413.500000  3413.800049    0.0  ...   
1994-01-12  3394.800049  3402.399902  3372.000000  3372.000000    0.0  ...   
1994-01-13  3380.699951  3383.300049  3356.899902  3360.000000    0.0  ...   

Ticker            ^GSPC       ^GDAXI                                         \
Price            Volume         Open         High          Low        Close   
Date                                                                          
1994-01-07  324920000.0  2218.959961  2227.639893  2201.820068  2224.949951   
1994-01-10  319490000.0  2231.840088  2238.010010  2222.000000  2225.000000   
1994-01-11  305490000.0  2225.429932  2235.610107  2225.179932  2228.100098   
1994-01-12  310690000.0  2227.120117  2227.790039  2182.060059  2182.060059   
1994-01-13  277970000.0  2171.500000  2183.709961  2134.100098  2142.370117   

Ticker                    spx          dax         ftse        nikkei  
Price      Volume                                                      
Date                                                                   
1994-01-07    0.0  469.899994  2224.949951  3446.000000  18124.009766  
1994-01-10    0.0  475.269989  2225.000000  3440.600098  18443.439453  
1994-01-11    0.0  474.130005  2228.100098  3413.800049  18485.250000  
1994-01-12    0.0  474.170013  2182.060059  3372.000000  18793.880859  
1994-01-13    0.0  472.470001  2142.370117  3360.000000  18577.259766  

[5 rows x 24 columns]

In [7]:
df_comp.tail()

Ticker             ^N225                                            \
Price               Open          High           Low         Close   
Date                                                                 
2020-03-13  18183.470703  18184.460938  16690.599609  17431.050781   
2020-03-16  17586.080078  17785.759766  16914.449219  17002.039062   
2020-03-17  16726.949219  17557.039062  16378.940430  17011.529297   
2020-03-18  17154.080078  17396.839844  16698.460938  16726.550781   
2020-03-19  16995.769531  17160.970703  16358.190430  16552.830078   

Ticker                         ^FTSE                                         \
Price            Volume         Open         High          Low        Close   
Date                                                                          
2020-03-13  233400000.0  5237.500000  5696.500000  5237.500000  5366.100098   
2020-03-16  158100000.0  5366.100098  5366.100098  4898.799805  5151.100098   
2020-03-17  198800000.0  5151.100098  5309.000000  4978.799805  5294.899902   
2020-03-18  177200000.0  5294.899902  5294.899902  5006.200195  5080.600098   
2020-03-19  198900000.0  5080.600098  5181.000000  4942.399902  5151.600098   

Ticker                    ...         ^GSPC       ^GDAXI               \
Price             Volume  ...        Volume         Open         High   
Date                      ...                                           
2020-03-13  1.935105e+09  ...  8.299070e+09  9480.780273  9985.740234   
2020-03-16  2.111568e+09  ...  7.805450e+09  8728.480469  8967.110352   
2020-03-17  1.941165e+09  ...  8.370250e+09  9141.169922  9145.929688   
2020-03-18  1.837075e+09  ...  8.799300e+09  8613.349609  8670.410156   
2020-03-19  1.963412e+09  ...  7.956100e+09  8495.940430  8668.480469   

Ticker                                                     spx          dax  \
Price               Low        Close       Volume                             
Date                                                                          
2020-03-13  9064.679688  9232.080078  325900900.0  2711.020020  9232.080078   
2020-03-16  8255.650391  8742.250000  302202400.0  2386.129883  8742.250000   
2020-03-17  8423.559570  8939.099609  220092600.0  2529.189941  8939.099609   
2020-03-18  8400.179688  8441.709961  207558900.0  2398.100098  8441.709961   
2020-03-19  8257.530273  8610.429688  205539300.0  2409.389893  8610.429688   

Ticker             ftse        nikkei  
Price                                  
Date                                   
2020-03-13  5366.100098  17431.050781  
2020-03-16  5151.100098  17002.039062  
2020-03-17  5294.899902  17011.529297  
2020-03-18  5080.600098  16726.550781  
2020-03-19  5151.600098  16552.830078  

[5 rows x 24 columns]

In [8]:
del df_comp['^N225']
del df_comp['^GSPC']
del df_comp['^GDAXI']
del df_comp['^FTSE']
df_comp=df_comp.asfreq('b')
df_comp=df_comp.fillna(method='ffill')

In [9]:
df_comp.head()

Ticker,spx,dax,ftse,nikkei
Price,,,,
Date,,,,
1994-01-07,469.899994,2224.949951,3446.000000,18124.009766
1994-01-10,475.269989,2225.000000,3440.600098,18443.439453
1994-01-11,474.130005,2228.100098,3413.800049,18485.250000
1994-01-12,474.170013,2182.060059,3372.000000,18793.880859
1994-01-13,472.470001,2142.370117,3360.000000,18577.259766


### Creating Returns

In [10]:
df_comp['ret_spx'] = df_comp.spx.pct_change(1)*100
df_comp['ret_ftse'] = df_comp.ftse.pct_change(1)*100
df_comp['ret_dax'] = df_comp.dax.pct_change(1)*100
df_comp['ret_nikkei'] = df_comp.nikkei.pct_change(1)*100

### Splitting the Data

In [11]:
size = int(len(df_comp)*0.8)
df, df_test = df_comp.iloc[:size], df_comp.iloc[size:]

### Fitting a Model

In [2]:
pip uninstall pmdarima numpy -y

Found existing installation: pmdarima 2.0.4
Uninstalling pmdarima-2.0.4:
  Successfully uninstalled pmdarima-2.0.4
Found existing installation: numpy 2.3.2
Uninstalling numpy-2.3.2:
  Successfully uninstalled numpy-2.3.2
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


In [3]:
!pip install pmdarima


  Using cached pmdarima-2.0.4-cp312-cp312-win_amd64.whl.metadata (8.0 kB)
  Using cached numpy-2.3.2-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached pmdarima-2.0.4-cp312-cp312-win_amd64.whl (625 kB)
Using cached numpy-2.3.2-cp312-cp312-win_amd64.whl (12.8 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   

In [12]:
model_auto = pm.auto_arima(df.ret_ftse[1:])

In [13]:
model_auto

,order,"(4, ...)"
,seasonal_order,"(0, ...)"
,start_params,None
,method,'lbfgs'
,maxiter,50
,suppress_warnings,True
,out_of_sample_size,0
,scoring,'mse'
,scoring_args,{}
,trend,None
,with_intercept,True


In [14]:
model_auto.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                      y   No. Observations:                 5467
Model:               SARIMAX(4, 0, 5)   Log Likelihood               -8450.890
Date:                Fri, 25 Jul 2025   AIC                          16923.780
Time:                        02:16:51   BIC                          16996.451
Sample:                    01-10-1994   HQIC                         16949.135
                         - 12-23-2014                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.0291      0.023      1.268      0.205      -0.016       0.074
ar.L1          0.0161      0.080      0.201      0.841      -0.141       0.173
ar.L2         -0.6498      0.077     -8.429      0.000      -0.801      -0.499
ar.L3         -0.1685      0.071     -2.371      0.018      -0.308      -0.029
ar.L4          0.2135      0.075      2.855      0.004       0.067       0.360
ma.L1         -0.0389      0.080     -0.486      0.627      -0.196       0.118
ma.L2          0.6044      0.078      7.764      0.000       0.452       0.757
ma.L3          0.0750      0.069      1.088      0.277      -0.060       0.210
ma.L4         -0.2088      0.074     -2.832      0.005      -0.353      -0.064
ma.L5         -0.0995      0.009    -11.128      0.000      -0.117      -0.082
sigma2         1.2889      0.013     96.322      0.000       1.263       1.315
===================================================================================
Ljung-Box (L1) (Q):                   0.00   Jarque-Bera (JB):              7481.01
Prob(Q):                              0.98   Prob(JB):                         0.00
Heteroskedasticity (H):               1.81   Skew:                            -0.19
Prob(H) (two-sided):                  0.00   Kurtosis:                         8.72
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

ARMA(4,5)

### Important Arguments

In [15]:
model_auto2 = pm.auto_arima(df_comp.ret_ftse[1:], X = df_comp[['ret_spx', 'ret_dax', 'ret_nikkei']][1:], m = 5,
                       max_order = None, max_p = 7, max_q = 7, max_d = 2, max_P = 4, max_Q = 4, max_D = 2,
                       maxiter = 50, alpha = 0.05, n_jobs = -1, trend = 'ct', information_criterion = 'oob',
                       out_of_sample_size = int(len(df_comp)*0.2))


# exogenous -> outside factors (e.g other time series)
# m -> seasonal cycle length
# max_order -> maximum amount of variables to be used in the regression (p + q)
# max_p -> maximum AR components
# max_q -> maximum MA components
# max_d -> maximum Integrations
# maxiter -> maximum iterations we're giving the model to converge the coefficients (becomes harder as the order increases)
# alpha -> level of significance, default is 5% (0.05), which we should be using most of the time
# n_jobs -> how many models to fit at a time (-1 indicates "as many as possible")
# trend -> La tendencia "ct" usually (constant and trend), "c" (constant only), "t" (trend only), "n" (none) 
# information_criterion -> 'aic', 'aicc', 'bic', 'hqic', 'oob' 
#        (Akaike Information Criterion, Corrected Akaike Information Criterion,
#        Bayesian Information Criterion, Hannan-Quinn Information Criterion, or
#        "out of bag"--for validation scoring--respectively)
# out_of_smaple_size -> validates the model selection (pass the entire dataset, and set 20% to be the out_of_sample_size)

Cuando agregamos variables exogenas debemos asegurarnos que las variables exogenas y endogenas sean del mismo tipo, es decir si la varible endogena es estacionaria, las variables exogenas deben ser estacionarias.

In [16]:
model_auto2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                                  y   No. Observations:                 6834
Model:             SARIMAX(0, 0, 1)x(4, 0, [], 5)   Log Likelihood               -6809.059
Date:                            Fri, 25 Jul 2025   AIC                          13640.119
Time:                                    02:22:17   BIC                          13715.245
Sample:                                         0   HQIC                         13666.037
                                           - 6834                                         
Covariance Type:                              opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     -0.0007      0.015     -0.048      0.961      -0.031       0.029
drift      -2.425e-06   4.33e-06     -0.560      0.576   -1.09e-05    6.07e-06
x1             0.0871      0.006     15.202      0.000       0.076       0.098
x2             0.5619      0.005    110.682      0.000       0.552       0.572
x3             0.0729      0.004     16.659      0.000       0.064       0.081
ma.L1         -0.1156      0.008    -15.354      0.000      -0.130      -0.101
ar.S.L5       -0.0357      0.009     -4.051      0.000      -0.053      -0.018
ar.S.L10      -0.0495      0.010     -5.134      0.000      -0.068      -0.031
ar.S.L15      -0.0342      0.009     -3.773      0.000      -0.052      -0.016
ar.S.L20      -0.0234      0.009     -2.535      0.011      -0.041      -0.005
sigma2         0.4641      0.005     99.323      0.000       0.455       0.473
===================================================================================
Ljung-Box (L1) (Q):                   3.12   Jarque-Bera (JB):             15213.83
Prob(Q):                              0.08   Prob(JB):                         0.00
Heteroskedasticity (H):               0.48   Skew:                             0.22
Prob(H) (two-sided):                  0.00   Kurtosis:                        10.30
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""